[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 Easy: Gradient Norm Clipping

Implement **gradient norm clipping** — a training stability technique.

### Signature
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # Clip gradients in-place so total norm <= max_norm
    # Returns the original (unclipped) total norm
```

### Algorithm
1. Compute total norm: `sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. If total > max_norm: scale all grads by `max_norm / total`
3. Return original total norm

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch

## Gradient Clipping
학습을 하다보면 gradient가 갑자기 엄청 커지는 경우가 생김

**왜냐? backward() (역전파)할때 gradient가 레이어를 통과하면서 계속 곱해짐. 레이어가 깊어질수록 곱이 누적되기 때문에 기하급수적으로 커짐.**

ex. 이상한 데이터가 섞여있는 경우, loss가 이전에는 0.5, 0.6 이랬다가 갑자기 500 될수있음. 이런 경우 gradient exploding!

-> 그럼 weight가 한번에 너무 많이 업데이트돼서 학습이 망가짐.
= gradient exploding

그래서 gradient가 일정 크기(max_norm) 을 넘으면, 방향은 유지하되 크기만 줄여줌!!

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def clip_grad_norm(parameters, max_norm):
    # compute total norm, clip if needed, return original norm

    # gradient가 없는 파라미터는 제외함
    # backward()를 안한 애들은 .grad가 None이라서 연산하면 에러남
    parameters = [p for p in parameters if p.grad is not None]

    # 전체 gradient 크기를 계산
    """
    p.grad.norm() - 파라미터 하나의 gradient 크기
    제곱해주고
    이걸 모든 파라미터에 대해 합산
    그 이후 전체에 루트 씌우기
    """
    total_norm = torch.sqrt(sum(p.grad.norm() ** 2 for p in parameters))

    # +1e-6 해주는이유 - total_norm이 혹시라도 0일때를 방지해서 0으로 나눠줌
    clip_coef = max_norm / (total_norm + 1e-6)


    """
    gradient가 작은데 괜히 키우면 안되니까 <1 일때만 실행함
    <1 인 경우 total_norm > max_norm. gradient가 너무 크니까 줄여야 함
    >1 인 경우 total_norm < max_norm. gradient 정상이니까 건드릴 필요 없음
    """
    if clip_coef < 1:
      # 파라미터 여러개일수있으니까 모두 줄여줘야 함
      for p in parameters:
        """
        mul 뒤에 붙은 _ 는 새 텐서를 만들지 않고, 값들을 직접 바꿔준다는 뜻!
        clip_coef를 곱해서 gradient 크기를 줄임
        """
        p.grad.mul_(clip_coef)

    # clipping 전 원래 norm을 반환함. gradient가 얼마나 컸었는지 확인용
    return total_norm.item()

In [ ]:
def clip_grad_norm(parameters, max_norm):
  parameters = [p for p in parameters if p.grad is not None]
  total_norm = torch.sqrt(sum(p.grad_norm() ** 2 for p in parameters))
  clip_coef = max_norm / (total_norm + 1e-6)
  if clip_coef < 1:
    for p in parameters:
      p.grad.mul_(clip_coef)
  return total_norm.item()

In [4]:
# 🧪 Debug
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
print('Before:', p.grad.norm().item())
orig = clip_grad_norm([p], max_norm=1.0)
print('After: ', p.grad.norm().item())
print('Original norm:', orig)

Before: 100.0
After:  0.9999999403953552
Original norm: 100.0


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_clipping')


🧪 Testing: Gradient Norm Clipping (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Clips to max_norm (1.7ms)
  ✅ [2/4] Returns original norm (0.7ms)
  ✅ [3/4] No change when norm < max_norm (2.1ms)
  ✅ [4/4] Preserves direction (11.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (16.3ms total)
  Progress saved. Run status() to see your dashboard.

